# División (Split) del dataset de perfiles completos

Con el fin de preparar los datos para el entrenamiento de modelos, en este notebook se genera una metadata persistente con una partición estratificada.

El objetivo es que otros notebooks puedan reutilizar exactamente la misma división `train` / `val` / `test` sin recalcularla.

El resultado se guarda como `metadata.csv` con una nueva columna `split`.

`CLASES_A_DIVIDIR` selecciona las clases que se procesan. Sus imágenes se copian a `DATA_DESTINO` conservando las carpetas de clase y las subcarpetas originales. La división `train` / `val` / `test` queda registrada en la metadata, sin crear carpetas por split. Al repetir la ejecución se sobrescriben los archivos coincidentes; no se eliminan archivos de ejecuciones anteriores.

In [1]:
# Importaciones
from vasijas.config import load_config
from vasijas.metadata import cargar_metadata, guardar_metadata
from vasijas.paths import ProjectPaths
from vasijas.split_dataset import (
    ConfiguracionSplitDataset,
    copiar_imagenes_split,
    preparar_metadata_split_perfiles_completos,
    describir_configuracion_split,
    advertir_imagenes_sin_metadata,
    mostrar_resumen_split,
    construir_resumen_split,
)

In [2]:
# Configuracion general
CONFIG = load_config()
PATHS = ProjectPaths.from_root(CONFIG.project_root)

## Origen y destino de datos

In [3]:
# Origen y destino de datos
DATA_ORIGEN = PATHS.DATA_CUADRADOS
DATA_DESTINO = PATHS.DATA_DIVIDIDO
print(f"Dataset de imágenes a dividir: {DATA_ORIGEN}")
print(f"Destino de imágenes y archivo de metadata: {DATA_DESTINO}")

Dataset de imágenes a dividir: C:\Users\rafae\Dev\vasijas\data\05.cuadrados
Destino de imágenes y archivo de metadata: C:\Users\rafae\Dev\vasijas\data\10.cuadrados_divididos_en_train_val_y_test


## Vista previa al procesamiento

In [4]:
METADATA_ORIGEN = cargar_metadata(DATA_ORIGEN)
print("Muestra metadata origen:")
METADATA_ORIGEN.head()

Muestra metadata origen:


,Identificador,Tiene asa,Tendencia Formal,Borde,Cuello,Cuerpo,Base,Clase Ana,Diametro Borde,Diametro Base,Altura Total,Cronologia,Ubicacion
0,al_03a,No,II,1,B,2,2,11.0,182,54,53,ibero,Almedinilla
1,al_03b,No,II,1,B,2,2,11.0,178,57,46,ibero,Almedinilla
2,al_03c,No,II,1,B,2,2,11.0,185,56,49,ibero,Almedinilla
3,al_03d,No,II,1,B,2,2,11.0,160,41,45,ibero,Almedinilla
4,al_03e,No,II,1,B,2,2,11.0,164,54,44,ibero,Almedinilla


## Configuración del split

In [5]:
# Parámetros del split

SEED = 42
CLASES_A_DIVIDIR = [*range(1, 12)]

config_split = ConfiguracionSplitDataset(semilla=SEED)

print(describir_configuracion_split(config_split))

Split configurado -> train: 70%, val: 15%, test: 15%, semilla: 42


## Construcción del split

Se usa `torchvision.datasets.ImageFolder` dentro del módulo para respetar el mismo orden de muestras y las mismas clases que consume el entrenamiento. `CLASES_A_DIVIDIR` permite limitar el split a un subconjunto de carpetas/clases. Luego se aplica `train_test_split` dos veces: primero `train` contra el conjunto temporal, y después `val` contra `test`.


In [6]:
resultado_split = preparar_metadata_split_perfiles_completos(
    data_origen=DATA_ORIGEN,
    metadata_origen=METADATA_ORIGEN,
    configuracion_split=config_split,
    clases_a_dividir=CLASES_A_DIVIDIR,
)

df_split = resultado_split.registros_split

mostrar_resumen_split(resultado_split)

ValueError: Las siguientes clases no existen en DATA_ORIGEN: ['1', '10', '11', '2', '3', '4', '5', '6', '7', '8', '9']. Clases disponibles: ['clase01', 'clase02', 'clase03', 'clase04', 'clase05', 'clase06', 'clase07', 'clase08', 'clase09', 'clase10', 'clase11', 'sin_clase']

In [ ]:
# Tabla de archivos con split
df_split.head()

## Unión con metadata

La metadata se limita a las imágenes procesadas y se agregan las columnas `archivo`, `ruta_relativa`, `clase_carpeta` y `split`.


In [ ]:
df_metadata_split = resultado_split.metadata
advertir_imagenes_sin_metadata(resultado_split.ids_sin_metadata)
df_metadata_split.head()

In [ ]:
# Control de distribucion por split y clase
resumen_split = construir_resumen_split(df_metadata_split)
resumen_split

In [ ]:
# Copia de imágenes seleccionadas y guardado de metadata
imagenes_copiadas = copiar_imagenes_split(
    data_origen=DATA_ORIGEN,
    data_destino=DATA_DESTINO,
    registros_split=df_split,
)

ruta_metadata_split = guardar_metadata(
    df=df_metadata_split,
    carpeta_destino=DATA_DESTINO,
)

print(f"Metadata con split guardada en: {ruta_metadata_split}")
print(f"Registros guardados: {len(df_metadata_split)}")
print(f"Imágenes copiadas: {imagenes_copiadas}")

## Uso desde otros notebooks

```python
import pandas as pd

from vasijas.config import load_config
from vasijas.paths import ProjectPaths

CONFIG = load_config()
PATHS = ProjectPaths.from_root(CONFIG.project_root)

metadata_split = pd.read_csv(PATHS.DATA_DIVIDIDO / "metadata.csv")
train_df = metadata_split[metadata_split["split"] == "train"]
val_df = metadata_split[metadata_split["split"] == "val"]
test_df = metadata_split[metadata_split["split"] == "test"]
```
